# Task 3 — Streaming Auto Loader: landing -> bronze (Delta)

In [0]:
from pyspark.sql.functions import current_timestamp, current_date

CATALOG    = "dbr_dev"
STORAGE_ACCOUNT = "dlspl21databricks"
CONTAINER  = "gabrielajaniszews786"
SCHEMA     = "gabrielajaniszews786_bronze"
BASE       = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net"
CHECKPOINT = f"{BASE}/_checkpoint/sensor_data" # Checkpoint
TARGET     = f"{CATALOG}.{SCHEMA}.sensor_data"

### Setting up stream consumer using Kafka

In [0]:
EH_NAMESPACE = "evhpl24databricks"
EH_NAME = "gabrielajaniszews786_eventhub"
EH_CONN_STR = dbutils.secrets.get("default2", "eventhub-con-str-gabriela")

BOOTSTRAP = f"{EH_NAMESPACE}.servicebus.windows.net:9093"
JAAS = f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="$ConnectionString" password="{EH_CONN_STR}";'

raw = (spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", BOOTSTRAP)
    .option("subscribe", EH_NAME)
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "PLAIN")
    .option("kafka.sasl.jaas.config", JAAS)
    .option("startingOffsets", "earliest")   # Reading from the earliest available data
    .load())


In [0]:
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

event_schema = (StructType()
    .add("event_id", StringType())
    .add("schema_version", IntegerType())
    .add("site_name", StringType())
    .add("country", StringType())
    .add("bidding_zone", StringType())
    .add("timestamp_utc", StringType())
    .add("consumption_kwh", DoubleType())
    .add("avg_power_kw", DoubleType())
    .add("pue", DoubleType())
)

parsed = (raw
    .select(
        from_json(col("value").cast("string"), event_schema).alias("e"),
        col("partition"), col("offset"), col("timestamp").alias("enqueued_ts"))
    .select("e.*", "partition", "offset", "enqueued_ts")
    .withColumn("ingestion_ts", current_timestamp()))



In [0]:
print(len(EH_CONN_STR))
print(repr(EH_CONN_STR[-3:]))

In [0]:
import socket
socket.create_connection(("evhpl24databricks.servicebus.windows.net", 9093), timeout=10)

In [0]:
query = (parsed.writeStream
   .format("delta")
   .option("checkpointLocation", CHECKPOINT)
   .option("mergeSchema","true")  
   .trigger(availableNow=True)                
   .toTable(TARGET))

In [0]:
spark.table(TARGET).count()

In [0]:
%sql

SELECT *
FROM gabrielajaniszews786_bronze.sensor_data

In [0]:
for q in spark.streams.active:
    q.stop()

In [0]:
# Checking active streams

for q in spark.streams.active:
    print(q.name, q.id, q.status)